# 4th Down Decision Model — Data Pull

Pulls 10 seasons (2016–2025, regular season + playoffs) of play-by-play data via `nflreadpy`, filters down to 4th-down plays, and builds a clean `decision` label (`go` / `punt` / `field_goal`) for each one.

In [1]:
import polars as pl
import nflreadpy as nfl

## Config

In [2]:
SEASONS = list(range(2016, 2026))  # last 10 completed seasons

# Plays that don't represent a real 4th-down decision: penalties/pre-snap
# dead balls (no_play), missing play type, and kneel-outs to run clock.
EXCLUDED_PLAY_TYPES = {"no_play", "qb_kneel", "qb_spike"}

DECISION_MAP = {
    "punt": "punt",
    "field_goal": "field_goal",
    "pass": "go",
    "run": "go",
}

KEEP_COLUMNS = [
    "game_id",
    "season",
    "season_type",
    "week",
    "posteam",
    "defteam",
    "qtr",
    "down",
    "ydstogo",
    "yardline_100",
    "game_seconds_remaining",
    "half_seconds_remaining",
    "score_differential",
    "play_type",
    "decision",
    "field_goal_result",
    "epa",
    "wp",
    "wpa",
    "desc",
]

## Pull play-by-play and filter to 4th downs

In [3]:
print(f"Pulling play-by-play for seasons {SEASONS[0]}-{SEASONS[-1]}...")
pbp = nfl.load_pbp(SEASONS)
pbp.shape

Pulling play-by-play for seasons 2016-2025...


(484254, 372)

In [4]:
fourth = (
    pbp.filter(pl.col("down") == 4)
    .filter(~pl.col("play_type").is_in(EXCLUDED_PLAY_TYPES))
    .filter(pl.col("play_type").is_not_null())
    .with_columns(
        pl.col("play_type").replace(DECISION_MAP).alias("decision")
    )
    .filter(pl.col("decision").is_in(["punt", "field_goal", "go"]))
    .select(KEEP_COLUMNS)
    .drop_nulls(subset=["ydstogo", "yardline_100", "score_differential", "game_seconds_remaining"])
)

print(f"4th-down plays kept: {fourth.shape[0]:,} rows, {fourth.shape[1]} columns")

4th-down plays kept: 39,367 rows, 20 columns


## Sanity check

In [5]:
fourth["decision"].value_counts().sort("count", descending=True)

decision,count
str,u32
"""punt""",22519
"""field_goal""",9790
"""go""",7058


In [6]:
fourth.head(10)

game_id,season,season_type,week,posteam,defteam,qtr,down,ydstogo,yardline_100,game_seconds_remaining,half_seconds_remaining,score_differential,play_type,decision,field_goal_result,epa,wp,wpa,desc
str,i32,str,i32,str,str,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,str
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BAL""","""BUF""",1.0,4.0,6.0,71.0,3412.0,1612.0,0.0,"""punt""","""punt""",null,-0.405048,0.471003,0.021992,"""(11:52) (Punt formation) 4-S.K…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",1.0,4.0,18.0,76.0,3282.0,1482.0,0.0,"""punt""","""punt""",null,0.224181,0.418961,0.014649,"""(9:42) (Punt formation) 6-C.Sc…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",1.0,4.0,11.0,48.0,3030.0,1230.0,0.0,"""punt""","""punt""",null,0.447958,0.475927,-0.007276,"""(5:30) (Punt formation) 6-C.Sc…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BAL""","""BUF""",1.0,4.0,15.0,32.0,2741.0,941.0,0.0,"""field_goal""","""field_goal""","""made""",1.797426,0.5298,0.06181,"""(:41) 9-J.Tucker 50 yard field…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",2.0,4.0,10.0,64.0,2587.0,787.0,-3.0,"""punt""","""punt""",null,0.480402,0.370587,-0.004405,"""(13:07) (Punt formation) 6-C.S…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",2.0,4.0,1.0,1.0,1986.0,186.0,-10.0,"""run""","""go""",null,2.859358,0.22806,0.112149,"""(3:06) 25-L.McCoy left guard f…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",3.0,4.0,1.0,37.0,1553.0,1553.0,-3.0,"""run""","""go""",null,2.62021,0.411699,0.092453,"""(10:53) (Shotgun) 5-T.Taylor s…"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BUF""","""BAL""",3.0,4.0,15.0,31.0,1423.0,1423.0,-3.0,"""field_goal""","""field_goal""","""missed""",-3.855208,0.358248,-0.085883,"""(8:43) (Field Goal formation) …"
"""2016_01_BUF_BAL""",2016,"""REG""",1,"""BAL""","""BUF""",3.0,4.0,2.0,53.0,1363.0,1363.0,3.0,"""punt""","""punt""",null,-1.136957,0.675675,-0.033029,"""(7:43) (Punt formation) 4-S.Ko…"


## Save

In [7]:
out_parquet = "../data/fourth_downs.parquet"
out_csv = "../data/fourth_downs.csv"
fourth.write_parquet(out_parquet)
fourth.write_csv(out_csv)
print(f"Saved to {out_parquet} and {out_csv}")

Saved to ../data/fourth_downs.parquet and ../data/fourth_downs.csv
